### Import packages and define paths

In [274]:
from pathlib import Path
from gensim.models.word2vec import Word2Vec
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [275]:
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
EMBEDDINGS_DIR = PROJECT_ROOT / "embeddings"
DATA_DIR = PROJECT_ROOT / "data/dictionaries"
OUTPUT_DIR = PROJECT_ROOT / "data/associational_gender_bias"

warmth_competence_lexicon_path = DATA_DIR / "Seed Dictionaries.csv"

lexicon = pd.read_csv(warmth_competence_lexicon_path)

In [276]:
warmth_competence_lexicon_path = DATA_DIR / "Seed Dictionaries.csv"

lexicon = pd.read_csv(warmth_competence_lexicon_path)

### Load embeddings

In [277]:
romance_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "romance.w2v"))
sci_fi_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "sci_fi.w2v"))
literary_fiction_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "literary_fiction.w2v"))

### Create average gender vectors

In [278]:
female_list = ['she', 'daughter', 'hers', 'her', 'mother', 'woman', 'girl', 'herself', 'female', 'sister', 'daughters', 'mothers', 'women', 'girls', 'females', 'sisters', 'aunt', 'aunts', 'niece', 'nieces']
male_list = ['he', 'son', 'his', 'him', 'father', 'man', 'boy', 'himself', 'male', 'brother', 'sons', 'fathers', 'men', 'boys', 'males', 'brothers', 'uncle', 'uncles', 'nephew', 'nephews']

In [279]:
# Function to compute the average vector
def compute_average_vector(model, word_list):
    vectors = [model.wv[word] for word in word_list if word in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        raise ValueError(f"None of the words in {word_list} exist in the model.")

In [280]:
def update_gender_vectors(model, male_words, female_words, genre_name, save_dir):
    # Save original vectors
    original_he = model.wv["he"].copy()
    original_she = model.wv["she"].copy()

    # Compute new vectors
    he_vector = compute_average_vector(model, male_words)
    she_vector = compute_average_vector(model, female_words)

    # Safe update
    model.wv["he"] = he_vector
    model.wv["she"] = she_vector

    # Compare with original
    he_sim = cosine_similarity([original_he], [model.wv["he"]])[0][0]
    she_sim = cosine_similarity([original_she], [model.wv["she"]])[0][0]
    print(f"{genre_name} — he similarity with original: {he_sim:.6f}")
    print(f"{genre_name} — she similarity with original: {she_sim:.6f}")

    # Save the updated model to a new file
    save_path = save_dir / f"{genre_name}_with_gender_vectors.w2v"
    model.save(str(save_path))
    print(f"Updated embeddings saved to {save_path}")

    return model

In [281]:
# Example usage for multiple genres
genres = {
    "romance": romance_embeddings,
    "sci_fi": sci_fi_embeddings,
    "literary_fiction": literary_fiction_embeddings
}

for genre_name, embeddings in genres.items():
    update_gender_vectors(embeddings, male_list, female_list, genre_name, EMBEDDINGS_DIR)


romance — he similarity with original: 0.626463
romance — she similarity with original: 0.590047
Updated embeddings saved to /Users/tildeidunsloth/Desktop/Thesis/embeddings/romance_with_gender_vectors.w2v
sci_fi — he similarity with original: 0.698136
sci_fi — she similarity with original: 0.636416
Updated embeddings saved to /Users/tildeidunsloth/Desktop/Thesis/embeddings/sci_fi_with_gender_vectors.w2v
literary_fiction — he similarity with original: 0.683366
literary_fiction — she similarity with original: 0.640648
Updated embeddings saved to /Users/tildeidunsloth/Desktop/Thesis/embeddings/literary_fiction_with_gender_vectors.w2v


In [282]:
updated_romance_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "romance_with_gender_vectors.w2v"))
updated_sci_fi_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "sci_fi_with_gender_vectors.w2v"))
updated_literary_fiction_embeddings = Word2Vec.load(str(EMBEDDINGS_DIR / "literary_fiction_with_gender_vectors.w2v"))

### Define target dimensions/word categories

In [283]:
high_sociability_words = lexicon[(lexicon["Dictionary"] == "Sociability") & (lexicon["Dir"] == "high")]["term"].tolist()
low_sociability_words = lexicon[(lexicon["Dictionary"] == "Sociability") & (lexicon["Dir"] == "low")]["term"].tolist()
high_morality_words = lexicon[(lexicon["Dictionary"] == "Morality") & (lexicon["Dir"] == "high")]["term"].tolist()
low_morality_words = lexicon[(lexicon["Dictionary"] == "Morality") & (lexicon["Dir"] == "low")]["term"].tolist()

high_agency_words = lexicon[(lexicon["Dictionary"] == "Agency") & (lexicon["Dir"] == "high")]["term"].tolist()
low_agency_words = lexicon[(lexicon["Dictionary"] == "Agency") & (lexicon["Dir"] == "low")]["term"].tolist()
high_ability_words = lexicon[(lexicon["Dictionary"] == "Ability") & (lexicon["Dir"] == "high")]["term"].tolist()
low_ability_words = lexicon[(lexicon["Dictionary"] == "Ability") & (lexicon["Dir"] == "low")]["term"].tolist()

In [284]:
# Remove duplicates within each combined list, preserving order
high_warmth     = list(dict.fromkeys(high_sociability_words + high_morality_words))
low_warmth      = list(dict.fromkeys(low_sociability_words + low_morality_words))
high_competence = list(dict.fromkeys(high_agency_words + high_ability_words))
low_competence  = list(dict.fromkeys(low_agency_words + low_ability_words))

In [285]:
len(high_competence), len(low_competence), len(high_warmth), len(low_warmth)

(68, 60, 74, 81)

### Check missing words and make specific category list for each genre depending on missing words

In [286]:
genre_embeddings = {
    "romance": romance_embeddings,
    "sci_fi": sci_fi_embeddings,
    "literary_fiction": literary_fiction_embeddings
}

word_lists = {
    "high_competence": high_competence,
    "low_competence": low_competence,
    "high_warmth": high_warmth,
    "low_warmth": low_warmth
}
# Function to create genre-specific filtered lists
def create_genre_specific_lists(genre_embeddings, word_lists):
    filtered_lists = {}
    missing_summary = {}
    
    for genre_name, emb in genre_embeddings.items():
        for category_name, words in word_lists.items():
            # Filter words present in this genre's embeddings
            filtered = [w for w in words if w in emb.wv]
            missing = [w for w in words if w not in emb.wv]
            
            # Key name format: category_genre
            key = f"{category_name}_{genre_name}"
            filtered_lists[key] = filtered
            missing_summary[key] = missing
            
    return filtered_lists, missing_summary


In [287]:
# Apply the function
filtered_genre_lists, missing_genre_words = create_genre_specific_lists(genre_embeddings, word_lists)

# access the filtered lists
high_competence_romance = filtered_genre_lists["high_competence_romance"]
high_competence_sci_fi = filtered_genre_lists["high_competence_sci_fi"]
high_competence_literary_fiction = filtered_genre_lists["high_competence_literary_fiction"]

low_competence_romance = filtered_genre_lists["low_competence_romance"]
low_competence_sci_fi = filtered_genre_lists["low_competence_sci_fi"]
low_competence_literary_fiction = filtered_genre_lists["low_competence_literary_fiction"]

high_warmth_romance = filtered_genre_lists["high_warmth_romance"]
high_warmth_sci_fi = filtered_genre_lists["high_warmth_sci_fi"]
high_warmth_literary_fiction = filtered_genre_lists["high_warmth_literary_fiction"]

low_warmth_romance = filtered_genre_lists["low_warmth_romance"]
low_warmth_sci_fi = filtered_genre_lists["low_warmth_sci_fi"]
low_warmth_literary_fiction = filtered_genre_lists["low_warmth_literary_fiction"]

# inspect missing words
for key, missing in missing_genre_words.items():
    if missing:
        print(f"{key}: {len(missing)} missing words -> {missing}")

high_competence_romance: 15 missing words -> ['fearlessness', 'assertiveness', 'unassertive', 'striver', 'industrious', 'self-confident', 'self-reliant', 'conscientious', 'motivated', 'autonomous', 'dominating', 'felicitous', 'shrewd', 'discriminating', 'insightful']
low_competence_romance: 30 missing words -> ['diffident', 'unassertiveness', 'insecure', 'inactive', 'doubtful', 'unenterprising', 'lethargic', 'undedicated', 'unadventurous', 'unmotivated', 'nonresilient', 'spiritless', 'submissive', 'meek', 'docile', 'incompetent', 'uncompetitive', 'unintelligent', 'stupidity', 'dumbness', 'uneducated', 'uncreative', 'unimaginative', 'undiscriminating', 'maladroit', 'unwise', 'ineffective', 'illogical', 'unperceptive', 'inept']
high_warmth_romance: 14 missing words -> ['sociability', 'sociable', 'likable', 'unreserved', 'forthcoming', 'trustworthiness', 'altruistic', 'selfless', 'benevolence', 'softhearted', 'virtuous', 'incorrupt', 'unprejudiced', 'beneficent']
low_warmth_romance: 41 mi

In [288]:
# Combine categories
all_categories_romance = {
    "high_competence": high_competence_romance,
    "low_competence": low_competence_romance,
    "high_warmth": high_warmth_romance,
    "low_warmth": low_warmth_romance
}

all_categories_sci_fi = {
    "high_competence": high_competence_sci_fi,
    "low_competence": low_competence_sci_fi,
    "high_warmth": high_warmth_sci_fi,
    "low_warmth": low_warmth_sci_fi
}

all_categories_literary_fiction = {
    "high_competence": high_competence_literary_fiction,
    "low_competence": low_competence_literary_fiction,
    "high_warmth": high_warmth_literary_fiction,
    "low_warmth": low_warmth_literary_fiction
}

### Calculate similarity

In [289]:
# Function to calculate the cosine similarity using gensim's similarity method
def calculate_similarity(embeddings, word1, word2):
    if word1 in embeddings.wv and word2 in embeddings.wv:
        return embeddings.wv.similarity(word1, word2)
    else:
        return None  # if any word is missing

In [290]:
data_romance = []
data_sci_fi = []
data_literary_fiction = []

In [291]:
for dimensions, target_words in all_categories_romance.items():
        for target_word in target_words:
            # Calculate similarity with "he" and "she"
            similarity_he = calculate_similarity(romance_embeddings, target_word, "he")
            similarity_she = calculate_similarity(romance_embeddings, target_word, "she")
            
            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she  # Gender bias as the difference
                
                data_romance.append({
                    "dimension": dimensions,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

In [292]:
for dimensions, target_words in all_categories_sci_fi.items():
        for target_word in target_words:
            # Calculate similarity with "he" and "she"
            similarity_he = calculate_similarity(sci_fi_embeddings, target_word, "he")
            similarity_she = calculate_similarity(sci_fi_embeddings, target_word, "she")
            
            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she  # Gender bias as the difference
                
                data_sci_fi.append({
                    "dimension": dimensions,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

In [293]:
for dimensions, target_words in all_categories_literary_fiction.items():
        for target_word in target_words:
            # Calculate similarity with "he" and "she"
            similarity_he = calculate_similarity(literary_fiction_embeddings, target_word, "he")
            similarity_she = calculate_similarity(literary_fiction_embeddings, target_word, "she")
            
            if similarity_he is not None and similarity_she is not None:
                gender_bias = similarity_he - similarity_she  # Gender bias as the difference
                
                data_literary_fiction.append({
                    "dimension": dimensions,
                    "word": target_word,
                    "similarity_with_male_vector": similarity_he,
                    "similarity_with_female_vector": similarity_she,
                    "gender_bias": gender_bias
                })

In [294]:
# Convert results into a pandas DataFrame
df_romance = pd.DataFrame(data_romance)
df_sci_fi = pd.DataFrame(data_sci_fi)
df_literary_fiction = pd.DataFrame(data_literary_fiction)

df_romance.to_csv( OUTPUT_DIR / "romance_associational_gender_bias.csv", index=False)
df_sci_fi.to_csv( OUTPUT_DIR / "sci_fi_associational_gender_bias.csv", index=False)
df_literary_fiction.to_csv( OUTPUT_DIR / "literary_fiction_associational_gender_bias.csv", index=False)

In [295]:
df_romance[["word", "gender_bias"]].sort_values(by="gender_bias", ascending=False).head(10)

,word,gender_bias
100,civil,0.092137
41,graceful,0.086343
27,smart,0.083074
156,rough,0.082923
162,bad,0.074512
164,wrong,0.069599
169,liar,0.060067
66,dominated,0.059336
55,lazy,0.059258
161,unfair,0.054283


In [296]:
df_sci_fi[["word", "gender_bias"]].sort_values(by="gender_bias", ascending=False).head(10)

,word,gender_bias
175,liar,0.084542
2,confidence,0.083803
174,thief,0.061575
40,capable,0.059505
52,brilliant,0.057114
35,skilled,0.055417
76,ignorance,0.055131
187,unreliable,0.054015
88,inability,0.051107
150,unpleasant,0.050932


In [297]:
df_literary_fiction[["word", "gender_bias"]].sort_values(by="gender_bias", ascending=False).head(10)

,word,gender_bias
144,reliable,0.097959
179,cunning,0.094706
65,helpless,0.090493
125,tolerance,0.085208
124,tolerant,0.078179
1,confident,0.074827
34,skilled,0.067361
47,wisdom,0.064067
2,confidence,0.063271
50,logical,0.061905
